# Learn2Design 2026: A100 submission selection

Run all on a separate A100 40 GB Colab runtime. This notebook uses fresh processes for each seed, saves results in Drive, and never uploads to the competition portal. Set paths in the next cell. The internal experiment cutoff is 30 September 2026, 12:00 IST.

In [ ]:
KIT_PATH = ""
OUTPUT_DIR = "/content/drive/MyDrive/L2D_2026_selection"
PORTAL_ZIP = ""
PORTAL_SHA256 = ""
PORTAL_SOURCE_VERIFIED = False
KIT_SHA256 = "09f867c07287acf7486d2f6ab56a5edb905ea599e010aa93ad9250a079e733c9"
MICROBATCH = 2

## Setup

Use `dist/l2d_selection_kit.zip` from this project. Put it in Drive and set `KIT_PATH`, or leave the path empty and select it in the upload dialog. `PORTAL_ZIP` is optional; set it to the exact downloaded submission #250 archive if available.

In [ ]:
import hashlib, json, os, pathlib, subprocess, sys, time, zipfile
from google.colab import drive, files
drive.mount("/content/drive")
out = pathlib.Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)
if not KIT_PATH:
    up = files.upload()
    assert len(up) == 1
    KIT_PATH = "/content/" + next(iter(up))
kit = pathlib.Path(KIT_PATH)
assert kit.is_file(), kit
def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for part in iter(lambda: f.read(1048576), b""):
            h.update(part)
    return h.hexdigest()
if KIT_SHA256:
    assert sha(kit) == KIT_SHA256, "kit hash mismatch"
root = pathlib.Path("/content/l2d_selection")
root.mkdir(exist_ok=True)
with zipfile.ZipFile(kit) as z:
    z.extractall(root)
manifest = json.loads((root / "manifest.json").read_text())
for name, expected in manifest["sha256"].items():
    assert sha(root / name) == expected, "bundle file hash mismatch: " + name
base = pathlib.Path(PORTAL_ZIP) if PORTAL_ZIP else root / "artifacts/baselines/local_champion.zip"
assert base.is_file(), base
if PORTAL_SHA256:
    assert sha(base) == PORTAL_SHA256, "portal archive hash mismatch"
provenance = ("portal submission #250, source verified" if PORTAL_ZIP and PORTAL_SOURCE_VERIFIED else "unverified local/portal copy")
print("Kit:", sha(kit))
print("Baseline:", base, sha(base), provenance)
print("Repair:", sha(root / "dist/champion_repair.zip"))
print("Rescue:", sha(root / "dist/n5_rescue.zip"))

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "jax", "jaxlib", "jax-cuda12-plugin", "jax-cuda12-pjrt"], check=False, stdout=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "dfbench[cuda12]==0.3.3", "optax", "h5py", "matplotlib", "scipy>=1.15"], check=True)
print("Dependencies installed. JAX stays out of this notebook kernel.")

## Diagnostics

The GPU clock starts here and is saved in Drive. Rerunning resumes completed, hash-matching runs only. Baseline measurements are labeled adapted if the exact archive cannot fit 40 GB unchanged.

In [ ]:
clock = out / "run_clock.json"
if not clock.exists():
    clock.write_text(json.dumps({"gpu_start_epoch": time.time()}))
start = json.loads(clock.read_text())["gpu_start_epoch"]
gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"], text=True).strip().splitlines()
assert len(gpu) == 1, gpu
name, total, free = [v.strip() for v in gpu[0].split(",")]
assert "A100" in name and 38000 <= int(total) <= 45000, gpu
assert int(free) >= 28000, "A100 memory is already occupied"
probe = "import jax,jax.numpy as jnp; x=jnp.logspace(0,1,10,dtype=jnp.float64).block_until_ready(); assert x.dtype==jnp.float64; assert jax.devices('gpu'); print(jax.__version__,jax.devices(),x.dtype)"
env = os.environ.copy()
env.update(JAX_ENABLE_X64="true", XLA_PYTHON_CLIENT_PREALLOCATE="false", MPLCONFIGDIR="/tmp/l2d-mpl")
pathlib.Path(env["MPLCONFIGDIR"]).mkdir(exist_ok=True)
subprocess.run([sys.executable, "-c", probe], check=True, env=env)
for arc in [base, root / "dist/champion_repair.zip", root / "dist/n5_rescue.zip"]:
    subprocess.run([sys.executable, str(root / "tools/official/validate_submission.py"), str(arc)], check=True)
print("GPU:", gpu[0], "hours remaining:", round((12*3600 - (time.time()-start))/3600, 2))

In [ ]:
import signal

def smoke(tag, arc, adapted=False):
    folder = out / "diagnostics" / tag
    folder.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, str(root / "tools/bench.py"), "--archive", str(arc), "--round", "2", "--indices", "0", "--minutes", "0.2", "--batch", str(MICROBATCH), "--single-dir", "--out", str(folder)]
    if adapted:
        cmd.append("--adapt-baseline")
    remaining = min(600, 1800 - int(time.time()-start))
    if remaining < 60:
        print(tag, "skipped: diagnostic time cap")
        return {}, ""
    timed_out = False
    with (folder / "driver.log").open("w") as f:
        p = subprocess.Popen(cmd, cwd=root, stdout=f, stderr=subprocess.STDOUT, start_new_session=True, env=env)
        try:
            p.wait(timeout=remaining)
        except subprocess.TimeoutExpired:
            timed_out = True
            os.killpg(p.pid, signal.SIGTERM)
            try:
                p.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(p.pid, signal.SIGKILL)
                p.wait()
    txt = "\n".join((folder / x).read_text(errors="replace")[-6000:] for x in ("driver.log", "stderr.log") if (folder / x).exists())
    result = json.loads((folder / "run.json").read_text()) if (folder / "run.json").exists() else {}
    if timed_out:
        result["status"] = "timeout"
    print(tag, "exit", p.returncode, "status", result.get("status"), "loss", result.get("best_feasible_loss"))
    if result.get("error"):
        print(result["error"][:600])
    return result, txt

base_smoke, base_text = smoke("baseline_original", base)
adapt = any(x in base_text.lower() for x in ("resource_exhausted", "out of memory", "cuda_error_out_of_memory", "allocator-allocation-error")) or base_smoke.get("status") == "timeout"
if adapt:
    smoke("baseline_adapted", base, True)
smoke("champion_repair", root / "dist/champion_repair.zip")
smoke("n5_rescue", root / "dist/n5_rescue.zip")
print("Baseline adapted on A100:", adapt)

## Staged comparison and final selection

This cell can run for many hours. It checks the 12 GPU-hour limit and the 12:00 IST cutoff before each stage. Outputs and the selected ZIP go to `OUTPUT_DIR`.

In [ ]:
cmd = [sys.executable, str(root / "tools/choose_submission.py"), "--baseline", str(base), "--repair", str(root / "dist/champion_repair.zip"), "--rescue", str(root / "dist/n5_rescue.zip"), "--out", str(out), "--batch", str(MICROBATCH), "--baseline-provenance", provenance, "--smoke"]
if PORTAL_SHA256:
    cmd += ["--baseline-sha256", PORTAL_SHA256]
if adapt:
    cmd.append("--adapt-baseline")
with (out / "selection.log").open("a") as log:
    run = subprocess.run(cmd, cwd=root, stdout=log, stderr=subprocess.STDOUT, env=env)
print("Exit:", run.returncode)
print((out / "selection.log").read_text(errors="replace")[-6000:])

In [ ]:
from IPython.display import display, Markdown, Image
if (out / "decision.md").exists():
    display(Markdown((out / "decision.md").read_text()))
if (out / "comparison.png").exists():
    display(Image(filename=str(out / "comparison.png")))
if (out / "selected.zip").exists():
    decision = json.loads((out / "decision.json").read_text())
    print("UPLOAD THIS ONE ZIP:" if decision["upload_ready"] else "HOLD: validation is incomplete or failed:", out / "selected.zip")
    print("SHA-256:", sha(out / "selected.zip"))
    print("Validation:", (out / "validation.json").read_text())